In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [ ]:
DATASET_DIR = "/kaggle/input/datasets/daudshah/vehicle-detection-dataset"
TRAIN_DIR = os.path.join(DATASET_DIR, "train")
TEST_DIR = os.path.join(DATASET_DIR, "test")


def get_class_counts(data_dir):
    class_counts = {}
    if not os.path.exists(data_dir):
        return class_counts
    for class_name in os.listdir(data_dir):
        class_path = os.path.join(data_dir, class_name)
        if os.path.isdir(class_path):
            class_counts[class_name] = len(os.listdir(class_path))
    return class_counts


train_counts = get_class_counts("/kaggle/input/datasets/daudshah/vehicle-detection-dataset/train")
test_counts = get_class_counts("/kaggle/input/datasets/daudshah/vehicle-detection-dataset/test")

In [ ]:
## visualize data

In [ ]:

df_train = pd.DataFrame(
    list(train_counts.items()), columns=["Category", "Train_Count"]
)
df_test = pd.DataFrame(
    list(test_counts.items()), columns=["Category", "Test_Count"]
)

df_dist = pd.merge(df_train, df_test, on="Category", how="outer").fillna(0)
print(df_dist)

# Plot initial class distribution
plt.figure(figsize=(12, 6))
df_dist_melted = pd.melt(
    df_dist,
    id_vars=["Category"],
    value_vars=["Train_Count", "Test_Count"],
    var_name="Split",
    value_name="Count",
)
sns.barplot(
    data=df_dist_melted, x="Category", y="Count", hue="Split", palette="Set2"
)
plt.title("Initial Class Distribution (Train vs Test)")
plt.xticks(rotation=45)
plt.ylabel("Number of Images")
plt.tight_layout()
plt.show()

# Data Merging, Resizing, Normalization & Augmentation


In [ ]:

import hashlib
import cv2
import numpy as np

IMG_SIZE = (128, 128)


def load_and_merge_data(base_dir):
    """Loads, resizes, and normalizes all images from train and test sets."""
    images = []
    labels = []
    classes = sorted([
        for d in os.listdir(os.path.join(base_dir, "train"))
        if os.path.isdir(os.path.join(base_dir, "train", d))
    ])
    class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}

    for split in ["train", "test"]:
        split_path = os.path.join(base_dir, split)
        if not os.path.exists(split_path):
            continue
        for cls_name in classes:
            cls_path = os.path.join(split_path, cls_name)
            if not os.path.exists(cls_path):
                continue
            for img_name in os.listdir(cls_path):
                img_path = os.path.join(cls_path, img_name)
                img = cv2.imread(img_path)
                if img is not None:
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    img = cv2.resize(img, IMG_SIZE)
                    images.append(img)
                    labels.append(class_to_idx[cls_name])

    X = np.array(images, dtype="float32") / 255.0
    y = np.array(labels)
    return X, y, classes

In [ ]:
"""Detects and removes identical images based on byte hashing."""
def remove_duplicate_images(X, y):
    seen_hashes = set()
    unique_X, unique_y = [], []
    duplicates_found = 0

    for img, label in zip(X, y):
        img_hash = hashlib.md5(img.tobytes()).hexdigest()

        if img_hash not in seen_hashes:
            seen_hashes.add(img_hash)
            unique_X.append(img)
            unique_y.append(label)
        else:
            duplicates_found += 1

    print(f"Total raw images loaded : {len(X)}")
    print(f"Duplicates removed     : {duplicates_found}")
    print(f"Unique images remaining: {len(unique_X)}")

    return np.array(unique_X, dtype="float32"), np.array(unique_y)

In [ ]:
#  Load and merge raw images
X_raw, y_raw, class_names = load_and_merge_data("/kaggle/input/datasets/daudshah/vehicle-detection-dataset")

#  Remove duplicate images BEFORE balancing or train-test split
X_unique, y_unique = remove_duplicate_images(X_raw, y_raw)

In [ ]:
from sklearn.utils import resample
from tensorflow.keras.preprocessing.image import ImageDataGenerator


datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest",
)

In [ ]:
"""Balances dataset using downsampling for majority classes and augmentation for minority classes."""
def balance_dataset(X, y, target_per_class=1000):
    X_balanced, y_balanced = [], []

    for cls in np.unique(y):
        cls_indices = np.where(y == cls)[0]
        X_cls, y_cls = X[cls_indices], y[cls_indices]
        num_samples = len(X_cls)

        if num_samples >= target_per_class:
            X_sel, y_sel = resample(
                X_cls,
                y_cls,
                replace=False,
                n_samples=target_per_class,
                random_state=42,
            )
        else:
            X_sel, y_sel = list(X_cls), list(y_cls)
            needed = target_per_class - num_samples
            aug_iter = datagen.flow(X_cls, batch_size=1)

            for _ in range(needed):
                X_sel.append(next(aug_iter)[0])
                y_sel.append(cls)

        X_balanced.extend(X_sel)
        y_balanced.extend(y_sel)

    return np.array(X_balanced, dtype="float32"), np.array(y_balanced)

In [ ]:

X_balanced, y_balanced = balance_dataset(
    X_unique, y_unique, target_per_class=1000
)

In [ ]:
# Visualize balanced distribution
plt.figure(figsize=(10, 4))
unique_cls, counts = np.unique(y_balanced, return_counts=True)
sns.barplot(
    x=[class_names[i] for i in unique_cls], y=counts, palette="viridis"
)
plt.title("Balanced Class Distribution Post-Augmentation")
plt.xlabel("Vehicle Category")
plt.ylabel("Sample Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
)

#  Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced,
    y_balanced,
    test_size=0.2,
    random_state=42,
    stratify=y_balanced,
)

num_classes = len(class_names)
batch_size = 32

In [ ]:

train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest",
)

train_generator = train_datagen.flow(
    X_train, y_train, batch_size=batch_size, shuffle=True
)

In [ ]:
#  CNN Architecture
def build_vehicle_cnn(input_shape=(128, 128, 3), num_classes=5):
    model = models.Sequential([
       
        layers.Conv2D(
            32, (3, 3), activation="relu", padding="same", input_shape=input_shape
        ),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(256, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax"),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


cnn_model = build_vehicle_cnn(
    input_shape=(128, 128, 3), num_classes=num_classes
)

In [ ]:
#  Define Callbacks
reduce_lr = ReduceLROnPlateau(
    monitor="val_accuracy",
    factor=0.1,
    patience=4,  
    min_delta=0.0001,
    verbose=1,
)

early_stop = EarlyStopping(
    monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1
)

checkpoint = ModelCheckpoint(
    filepath="best_CNNModel.keras", monitor="val_accuracy", save_best_only=True
)
       

In [ ]:
# Fit Model with Callbacks
CNN_History = cnn_model.fit(
    train_generator,
    epochs=60,  
    validation_data=(X_test, y_test),
    callbacks=[reduce_lr, early_stop, checkpoint],
)


cnn_model.save("cnn_model.h5")

In [ ]:
#  Accuracy & Loss Visualization
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
# Accuracy Curve
ax[0].plot(CNN_History.history["accuracy"], label="Train Accuracy", linewidth=2)
ax[0].plot(
    CNN_History.history["val_accuracy"],
    label="Val Accuracy",
    linestyle="-",
    linewidth=2,
)
ax[0].set_title("Model Accuracy Curve")
ax[0].set_xlabel("Epochs")
ax[0].set_ylabel("Accuracy")
ax[0].legend()

# Loss Curve
ax[1].plot(CNN_History.history["loss"], label="Train Loss", linewidth=2)
ax[1].plot(
    CNN_History.history["val_loss"],
    label="Val Loss",
    linestyle="-",
    linewidth=2,
)
ax[1].set_title("Model Loss Curve")
ax[1].set_xlabel("Epochs")
ax[1].set_ylabel("Loss")
ax[1].legend()

plt.tight_layout()
plt.show()